In [ ]:
import copy
import json
import os
import sys
sys.path.append(os.path.dirname(os.path.dirname(os.getcwd())))

In [ ]:
# Inductive PU bagging at several negative sampling fractions.
# The groups come from the fixed sampler, which draws negatives from unlabeled sequences only.
hypothesis = 'ensemble_inductive_pu_learning_negative_fractions'
iteration = 50
negative_sizes = [0.1, 0.3, 0.6, 1.0]
submodels_num = 50

In [ ]:
base_configuration = {
    'hypothesis': hypothesis,
    'experiment': {
        'experiment_type': 'ensemble_submodel',
        'name': None,
        'kwargs': {}
    },
    'data': {
        'name': 'ensemble_submodel',
        'path': 'datasets/data_for_training/folds_training_dicts.pkl',
        'kwargs': {
            'ensemble_dict': {
                'algorithm_name': 'inductive_pu_learning',
                'iteration': iteration,
                'negative_size': None,
                'groups_path': None,
                'submodel_index': None,
            }
        }
    },
    'model': {
        'name': 'esm2_with_LA_head',
        'kwargs': {
            'model_name': 'facebook/esm2_t6_8M_UR50D',
            'num_labels': 1,
            'dout': 128,
            'kernel_size': 7,
            'use_max': True,
            'lora_kwargs': {
                'r': 10,
                'lora_alpha': 8,
                'lora_dropout': 0.3,
                'modules_to_save': ['classifier', 'light_attention'],
                'target_modules': [
                    r'(esm|model)\.encoder\.layer\.\d+\.attention\.self\.query',
                    r'(esm|model)\.encoder\.layer\.\d+\.attention\.self\.value'
                ]
            }
        },
    },
    'compile': {
        'optimizer': {
            'name': 'adamw',
            'kwargs': {'learning_rate': 0.001, 'weight_decay': 0.01}
        },
        'loss': {
            'name': 'binary_cross_entropy',
            'kwargs': {}
        },
        'trainer': {
            'name': 'regular',
            'kwargs': {}
        },
        'training_arguments': {
            'name': 'regular',
            'kwargs': {
                'epochs': 200,
                'batch_size': 256,
                'metric': 'roc_auc',
            }
        }
    },
    'train': {
        'name': 'LM'
    }
}

In [ ]:
dir_path = f'../data/{hypothesis}'
os.makedirs(dir_path, exist_ok=True)

for negative_size in negative_sizes:
    for index in range(submodels_num):
        experiment = f'fraction_{negative_size}_index_{index}'
        training_configuration = copy.deepcopy(base_configuration)
        training_configuration['experiment']['name'] = experiment
        ensemble_dict = training_configuration['data']['kwargs']['ensemble_dict']
        ensemble_dict['negative_size'] = negative_size
        ensemble_dict['groups_path'] = f'datasets/data_for_training/ensembles/inductive_pu_learning/iteration_{iteration}_negative_size_{negative_size}'
        ensemble_dict['submodel_index'] = index
        with open(os.path.join(dir_path, f'{experiment}.json'), 'w') as f:
            json.dump(training_configuration, f)